# PSInSAR Mine Landslide Detection — Full Pipeline

This notebook walks you through the complete PSInSAR workflow:

1. **Data Acquisition** — Search Sentinel-1 SLC data via ASF, submit InSAR jobs to HyP3
2. **Preprocessing** — Coregistration, interferogram stacking
3. **PS Estimation** — Identify stable reflectors, remove atmospheric noise
4. **Velocity Generation** — Compute deformation velocity maps
5. **Visualization** — Interactive maps, time series, alerts

### Before you start:
- [Register for NASA Earthdata](https://urs.earthdata.nasa.gov/) (free)
- Set up credentials: `echo 'machine urs.earthdata.nasa.gov login USERNAME password PASSWORD' >> ~/.netrc`
- Edit `config.yaml` with your mine bounding box

In [6]:
# Install dependencies
#!pip install hyp3-sdk asf-search mintpy rasterio geopandas folium plotly scikit-learn

import sys
sys.path.insert(0, '../src')

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import os
from pathlib import Path
import yaml

# This finds 'psinsar_mine' no matter where the notebook is running
current_path = Path().absolute()
root_dir = current_path
while root_dir.name != 'psinsar_mine' and root_dir.parent != root_dir:
    root_dir = root_dir.parent

config_path = root_dir / 'config.yaml'

if not config_path.exists():
    raise FileNotFoundError(f" Still can't find config.yaml at: {config_path}")

with open(config_path) as f:
    cfg = yaml.safe_load(f)

# Set working directory to root so imports and data paths work correctly
os.chdir(root_dir)

print(f" Successfully loaded config from: {config_path}")
print(f"Mine: {cfg['aoi']['name']}")
print(f"AOI: {cfg['aoi']['bbox']}")

 Successfully loaded config from: c:\Users\Nirupon\Desktop\MineSafeV2\psinsar_mine\config.yaml
Mine: MineSafe
AOI: [82.53, 22.3, 82.66, 22.36]


In [8]:
!pip install hyp3-sdk asf-search mintpy rasterio geopandas folium plotly scikit-learn

  Using cached hyp3_sdk-7.7.6-py3-none-any.whl.metadata (6.0 kB)
  Using cached asf_search-12.0.1-py3-none-any.whl.metadata (11 kB)
  Using cached geopandas-1.1.2-py3-none-any.whl.metadata (2.3 kB)
  Using cached plotly-6.5.2-py3-none-any.whl.metadata (8.5 kB)
  Using cached dateparser-1.3.0-py3-none-any.whl.metadata (30 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached pyogrio-0.12.1-cp310-cp310-win_amd64.whl.metadata (6.0 kB)
  Using cached regex-2026.2.19-cp310-cp310-win_amd64.whl.metadata (41 kB)
  Using cached tzlocal-5.3.1-py3-none-any.whl.metadata (7.6 kB)
Using cached hyp3_sdk-7.7.6-py3-none-any.whl (15 kB)
Using cached asf_search-12.0.1-py3-none-any.whl (117 kB)
Using cached geopandas-1.1.2-py3-none-any.whl (341 kB)
Using cached plotly-6.5.2-py3-none-any.whl (9.9 MB)
Using cached pyogrio-0.12.1-cp310-cp310-win_amd64.whl (22.9 MB)
Using cached tenacity-9.1.4-py3-none-any.whl (28 kB)
Using cached dateparser-1.3.0-py3-none-any.whl (318 kB)
Using 

## Step 1: Search for Sentinel-1 Data

The Alaska Satellite Facility (ASF) hosts the complete Sentinel-1 archive for free.
We use `asf-search` to find all SLC acquisitions over our mine.

In [9]:
import asf_search as asf

bbox = cfg['aoi']['bbox']
wkt = (f"POLYGON(({bbox[0]} {bbox[1]}, {bbox[2]} {bbox[1]}, "
       f"{bbox[2]} {bbox[3]}, {bbox[0]} {bbox[3]}, {bbox[0]} {bbox[1]}))")

results = asf.search(
    platform=[asf.PLATFORM.SENTINEL1],
    processingLevel=[asf.PRODUCT_TYPE.SLC],
    beamMode=[asf.BEAMMODE.IW],
    flightDirection=cfg['sentinel1']['flight_direction'],
    start=cfg['time_range']['start'],
    end=cfg['time_range']['end'],
    intersectsWith=wkt,
)

print(f"Found {len(results)} Sentinel-1 SLC granules")
print(f"Date range covered: {results[0].properties['startTime'][:10]} to {results[-1].properties['startTime'][:10]}")

# Show first few
for r in results[:5]:
    print(f"  {r.properties['sceneName']}  |  {r.properties['startTime'][:10]}")

Found 172 Sentinel-1 SLC granules
Date range covered: 2024-12-30 to 2022-01-03
  S1A_IW_SLC__1SDV_20241230T002923_20241230T002950_057214_07096B_BF48  |  2024-12-30
  S1A_IW_SLC__1SDV_20241225T002117_20241225T002144_057141_070696_F40B  |  2024-12-25
  S1A_IW_SLC__1SDV_20241218T002924_20241218T002950_057039_07027F_F966  |  2024-12-18
  S1A_IW_SLC__1SDV_20241206T002925_20241206T002952_056864_06FB8D_567E  |  2024-12-06
  S1A_IW_SLC__1SDV_20241124T002926_20241124T002953_056689_06F4A4_D66B  |  2024-11-24


## Step 2: Submit InSAR Jobs to HyP3

HyP3 is NASA ASF's free cloud-processing platform for SAR data.
Each InSAR pair job generates:
- Unwrapped interferometric phase
- Coherence map
- DEM
- Look vectors & incidence angle map

**New accounts get 10,000 free credits/month** (~10,000 InSAR pairs).

In [18]:
import hyp3_sdk as sdk

hyp3 = sdk.HyP3()
user_info = hyp3.my_info()

# .get() provides a fallback if the key isn't found
user_name = user_info.get('username') or user_info.get('user_id') or "User Found"
credits = user_info.get('remaining_credits') or user_info.get('quota', {}).get('remaining', 0)

print(f"Connected as: {user_name}")
print(f"Remaining credits: {credits}")

Connected as: sayantanpatra
Remaining credits: 8000


In [13]:
from datetime import datetime

# Build SBAS pairs (each image paired with images within 48 days)
sorted_results = sorted(results, key=lambda g: g.properties['startTime'])
pairs = []
for i, ref in enumerate(sorted_results):
    ref_dt = datetime.fromisoformat(ref.properties['startTime'].replace('Z',''))
    for sec in sorted_results[i+1:]:
        sec_dt = datetime.fromisoformat(sec.properties['startTime'].replace('Z',''))
        if (sec_dt - ref_dt).days > 48:
            break
        pairs.append((ref, sec))

print(f"Generated {len(pairs)} SBAS pairs")
print(f"Estimated cost: ~{len(pairs)} credits")

# UNCOMMENT to actually submit:
batch = sdk.Batch()
for ref, sec in pairs:
    job = hyp3.submit_insar_job(
        granule1=ref.properties['sceneName'],
        granule2=sec.properties['sceneName'],
        name='MineLandslide',
        include_dem=True,
        include_look_vectors=True,
    )
    batch += sdk.Batch([job])
print(f'Submitted {len(batch)} jobs')

Generated 1266 SBAS pairs
Estimated cost: ~1266 credits


KeyboardInterrupt: 

## Step 3: Run PSInSAR Analysis with Demo Data

While waiting for HyP3 jobs (30-90 min), we can test the full pipeline with synthetic data.

In [ ]:
# Run demo pipeline (synthetic data)
import subprocess

output_dir = Path('../outputs')
output_dir.mkdir(exist_ok=True)

# Step 4 in demo mode generates synthetic velocity data
result = subprocess.run(
    ['python', '../src/04_velocity_generation.py', '--config', '../config.yaml', '--demo'],
    capture_output=True, text=True
)
print(result.stdout[-2000:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

In [ ]:
# Load and visualize
df = pd.read_csv('../outputs/ps_points_velocity.csv')
print(f"PS points: {len(df):,}")
print(df.describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Velocity map
sc = axes[0].scatter(df['lon'], df['lat'], c=df['velocity_mm_yr'],
                     cmap='RdYlBu', s=1, vmin=-30, vmax=30)
plt.colorbar(sc, ax=axes[0], label='mm/yr')
axes[0].set_title('LOS Velocity Map')
axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')

# Histogram
axes[1].hist(df['velocity_mm_yr'], bins=60, color='#4472C4', edgecolor='white')
thresh = cfg['output']['alert_threshold_mm_per_year']
axes[1].axvline(thresh, color='red', linestyle='--', label=f'±{thresh} mm/yr')
axes[1].axvline(-thresh, color='red', linestyle='--')
axes[1].set_xlabel('Velocity (mm/yr)'); axes[1].set_ylabel('Count')
axes[1].set_title('Velocity Distribution')
axes[1].legend()

plt.tight_layout()
plt.show()
print(f"\nAlert points: {df['alert'].sum():,} / {len(df):,}")

In [ ]:
# Run visualization step (creates interactive HTML map)
result = subprocess.run(
    ['python', '../src/05_visualization.py', '--config', '../config.yaml', '--demo'],
    capture_output=True, text=True
)
print(result.stdout[-3000:])

# Display the dashboard image
from IPython.display import Image, display
display(Image('../outputs/dashboard.png'))

## Next Steps

1. **Set your mine location** in `config.yaml`
2. **Register** at https://urs.earthdata.nasa.gov/
3. **Submit HyP3 jobs** using the cells above (uncomment)
4. **Wait ~1 hour** for processing
5. **Run** `python src/02_preprocessing.py` to prep the stack
6. **Run** `python src/03_ps_estimation.py` to run MintPy
7. **Run** `python src/04_velocity_generation.py` for velocity maps
8. **Run** `python src/05_visualization.py` for final outputs

### Key Resources
- [HyP3 Documentation](https://hyp3-docs.asf.alaska.edu/)
- [ASF Vertex Portal](https://search.asf.alaska.edu/) — visual interface for browsing Sentinel-1 data
- [MintPy Documentation](https://mintpy.readthedocs.io/)
- [GitHub: hyp3_insar landslide workflow](https://github.com/forrestfwilliams/hyp3_insar)